# sklearn Pipeline

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (StandardScaler, OrdinalEncoder, OneHotEncoder)
from sklearn.impute import SimpleImputer
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 50)
print("All libraries loaded successfully")

All libraries loaded successfully


In [2]:
# Load raw data and basic preparation
df = pd.read_csv('../data/raw/titanic.csv')

# Feature Creation
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)
df['Title'] = df['Name'].str.extract(r',\s([A-Za-z]+)\.')

# Group Rare titles
title_counts = df['Title'].value_counts()
common_titles = title_counts[title_counts >= 10].index
df['Title'] = df['Title'].apply(lambda x: x if x in common_titles else 'Rare')

# Drop Columns
df = df.drop(columns=['Name', 'Ticket', 'PassengerId'])

X = df.drop(columns=['Survived'])
y = df['Survived']

# Train Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set: {X_train.shape}")
print(f"Testing set: {X_test.shape}")
print("Columns:")
print(X_train.columns.tolist())

Training set: (712, 11)
Testing set: (179, 11)
Columns:
['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Cabin', 'Embarked', 'FamilySize', 'IsAlone', 'Title']


In [3]:
df.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Cabin,Embarked,FamilySize,IsAlone,Title
0,0,3,male,22.0,1,0,7.2500,NaN,S,2,0,Mr
1,1,1,female,38.0,1,0,71.2833,C85,C,2,0,Mrs
2,1,3,female,26.0,0,0,7.9250,NaN,S,1,1,Miss
3,1,1,female,35.0,1,0,53.1000,C123,S,2,0,Mrs
4,0,3,male,35.0,0,0,8.0500,NaN,S,1,1,Mr


In [4]:
# Fill Structural missing values

# Extract just the first letter of Cabin (the deck)
# 'C85' → 'C', 'B28' → 'B', 'No_Cabin' → 'N'
# This reduces ~100 unique cabins to just 9 deck letters
df['Cabin'] = df['Cabin'].fillna('No_Cabin')
df['Cabin'] = df['Cabin'].str[0]   # keep only first character

print("Cabin unique values after deck extraction:")
print(df['Cabin'].value_counts())

Cabin unique values after deck extraction:
Cabin
N    687
C     59
B     47
D     33
E     32
A     15
F     13
G      4
T      1
Name: count, dtype: int64


In [5]:
# Define Column Groups

# Numerical
numerical_cols = ['Age', 'Fare', 'FamilySize']
# Categorical
categorical_cols = ['Embarked', 'Title', 'Cabin']
# Binary 
binary_cols = ['Sex']
# Ordinal 
ordinal_cols = ['Pclass', 'IsAlone', 'SibSp', 'Parch']

print("Numerical:", numerical_cols)
print("Categorical:", categorical_cols)
print("Binary:", binary_cols)
print("Ordinal:", ordinal_cols)

all_defined_cols = numerical_cols + categorical_cols + binary_cols + ordinal_cols
missing_from_groups = [c for c in X_train.columns if c not in all_defined_cols]
print(f"Columns not assigned to any group: {missing_from_groups}")

Numerical: ['Age', 'Fare', 'FamilySize']
Categorical: ['Embarked', 'Title', 'Cabin']
Binary: ['Sex']
Ordinal: ['Pclass', 'IsAlone', 'SibSp', 'Parch']
Columns not assigned to any group: []


In [6]:
# Build Sub-pipelines for each column group
numerical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')), # Fill missing values with median
    ('scaler', StandardScaler()) # Scale to mean=0, std=1
])

categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')), # Fill missing values with most frequent value
    ('encoder', OneHotEncoder(handle_unknown='ignore', drop='first', sparse_output=False))
])

binary_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(handle_unknown = 'use_encoded_value', unknown_value= -1))
])

ordinal_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

print("Sub-pipelines built successfully")

Sub-pipelines built successfully


## ColumnTransformer
ColumnTransformer applies different transformations to different column groups simultaneously.

In [7]:
# Routes Each Column to Its Correct Sub-Pipeline
preprocessor = ColumnTransformer(transformers=[
    # Format: ('name', pipeline_object, list_of_columns)
    ('num', numerical_pipeline, numerical_cols),
    ('cat', categorical_pipeline, categorical_cols),
    ('bin', binary_pipeline, binary_cols),
    ('ord', ordinal_pipeline, ordinal_cols)
], remainder='drop')

print("ColumnTransformer built.")
print("Routes:")
print(f"numerical_pipeline - {numerical_cols}")
print(f"categorical_pipeline - {categorical_cols}")
print(f"binary_pipeline - {binary_cols}")
print(f"ordinal_pipeline - {ordinal_cols}")

ColumnTransformer built.
Routes:
numerical_pipeline - ['Age', 'Fare', 'FamilySize']
categorical_pipeline - ['Embarked', 'Title', 'Cabin']
binary_pipeline - ['Sex']
ordinal_pipeline - ['Pclass', 'IsAlone', 'SibSp', 'Parch']


In [8]:
# Fit on Train, Transform Both

preprocessor.fit(X_train)
print("Pipeline fitted on training data.")

X_train_processed = preprocessor.transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print(f"X_train_processed shape: {X_train_processed.shape}")
print(f"X_test_processed shape: {X_test_processed.shape}")
print(f"Input columns: {X_train.shape[1]}")
print(f"Output column: {X_train_processed.shape[1]}")
print("Output has more columns than input because OneHotEncoder expanded categorical columns")

Pipeline fitted on training data.
X_train_processed shape: (712, 130)
X_test_processed shape: (179, 130)
Input columns: 11
Output column: 130
Output has more columns than input because OneHotEncoder expanded categorical columns


In [9]:
# Save Pipeline and Processed Data

os.makedirs('../models', exist_ok=True)
os.makedirs('../data/processed', exist_ok=True)

# Save the fitted pipeline
joblib.dump(preprocessor, '../models/preprocessor.pkl')
print("Pipeline saved successfully")

# Save processed datasets for use in Module 3
pd.DataFrame(X_train_processed).to_csv('../data/processed/X_train.csv', index=False)
pd.DataFrame(X_test_processed).to_csv('../data/processed/X_test.csv', index=False)
y_train.to_csv('../data/processed/y_train.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)

print("Processed data saved successfully")

Pipeline saved successfully
Processed data saved successfully


In [10]:
# Loading and Using the Pipeline on New Data=

# Simulate a new passenger arriving in production
new_passenger = pd.DataFrame({
    'Age'       : [25],
    'Fare'      : [52.0],
    'FamilySize': [1],
    'Embarked'  : ['S'],
    'Title'     : ['Mr'],
    'Cabin'     : ['No_Cabin'],
    'Pclass'    : [3],
    'Sex'       : [0],
    'IsAlone'   : [1],
    'SibSp'     : [0],
    'Parch'     : [0]
})

# Load pipeline from disk
loaded_pipeline = joblib.load('../models/preprocessor.pkl')

# Transform the new passenger using training parameters
new_processed = loaded_pipeline.transform(new_passenger)

print("New passenger - raw values:")
print(new_passenger)
# print()
print(f"After pipeline (shape): {new_processed.shape}")
print("Model is ready")

New passenger - raw values:
   Age  Fare  FamilySize Embarked Title     Cabin  Pclass  Sex  IsAlone  \
0   25  52.0           1        S    Mr  No_Cabin       3    0        1   

   SibSp  Parch  
0      0      0  
After pipeline (shape): (1, 130)
Model is ready
